# Optimizer comparison on classification using Digits dataset and MLP

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor, Lambda, Compose

from src.quadratic_annealing_optimizer import QuadraticAnnealingOptimizer
from src.models import QuadraticMLP
from src.utils import data_load_and_prep, build_sampler
from src.training import train
from src.newton_optimizer import NewtonOptimizer

EXPERIMENT_NAME = "optimizer-comparison-mlp-MNIST-q"

In [2]:
# Loading MNIST dataset
transform = Compose([
    ToTensor(),
    Lambda(lambda x: x.view(-1)),
])
train_data = datasets.MNIST(root="../data", train=True, download=False, transform=transform)
test_data = datasets.MNIST(root="../data", train=False, download=False, transform=transform)

In [3]:
train_data

Dataset MNIST
    Number of datapoints: 60000
    Root location: ../data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Lambda()
           )

In [4]:
NUM_EPOCHS = 30
HIDDEN_SIZES = [128, 32]
# BATH_SIZE = 128
train_loader = DataLoader(train_data, batch_size=len(train_data), shuffle=True)
test_loader = DataLoader(test_data, batch_size=len(test_data), shuffle=False)

# Classical optimization for benchmarking

In [5]:
loss_fn = nn.CrossEntropyLoss()
adam_model = QuadraticMLP(784, HIDDEN_SIZES, 10)
classical_device = "cpu" 
adam_optimizer = torch.optim.Adam(adam_model.parameters(), 
                             lr=0.01,
                             betas=[0.9, 0.999],
                             )

## Optimization using Adam optimizaer

In [6]:
train(
    model=adam_model,
    train_loader=train_loader,
    test_loader=test_loader,
    loss_fn=loss_fn,
    optimizer=adam_optimizer,
    c_device=classical_device,
    epochs=NUM_EPOCHS, 
    experiment_name = EXPERIMENT_NAME,
    run_name="adam-optimizer"
)

/home/filip/studia/master/second-order-by-annealer/venv/lib/python3.12/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


Epoch 000 | train_loss=1.9715 | train_acc=0.310 | test_loss=1.9703 | test_acc=0.304 | 
Epoch 005 | train_loss=0.9486 | train_acc=0.784 | test_loss=0.9434 | test_acc=0.790 | 
Epoch 010 | train_loss=0.5540 | train_acc=0.865 | test_loss=0.5453 | test_acc=0.867 | 
Epoch 015 | train_loss=0.3890 | train_acc=0.892 | test_loss=0.3837 | test_acc=0.893 | 
Epoch 020 | train_loss=0.3125 | train_acc=0.909 | test_loss=0.3086 | test_acc=0.911 | 
Epoch 025 | train_loss=0.2637 | train_acc=0.924 | test_loss=0.2615 | test_acc=0.925 | 


2026/05/02 18:00:25 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


Epoch 029 | train_loss=0.2357 | train_acc=0.932 | test_loss=0.2367 | test_acc=0.931 | 


{'run_id': '646ef5a709af49a5862775d3d5f3b628',
 'experiment_id': '467419381031699499',
 'experiment_name': 'optimizer-comparison-mlp-MNIST-q',
 'tracking_uri': 'file:///home/filip/studia/master/second-order-by-annealer/mlruns',
 'warm_start_epochs': 1,
 'optimization_time_sec': 2.538175117000719,
 'raw_optimization_time_sec': 2.648327987000812,
 'sum_epoch_optimization_time_sec': 2.538175117000719,
 'train_evaluation_time_sec': 119.23281051599588,
 'test_evaluation_time_sec': 18.816482342002928,
 'evaluation_time_sec': 138.0492928579988,
 'training_time_sec': 258.94779600400034,
 'final_train_loss': 0.2357296198606491,
 'final_test_loss': 0.23673173785209656,
 'final_train_metric': 0.9319666666666667,
 'final_test_metric': 0.9306,
 'optimizer': 'Adam',
 'seed': None,
 'epochs_completed': 30,
 'early_stopped': False,
 'early_stop_reason': None,
 'zero_acceptance_patience': 6,
 'final_zero_acceptance_streak': 0}

## Optimization using Adam optimizaer on cuda

In [7]:
loss_fn = nn.CrossEntropyLoss()
adam_model = QuadraticMLP(784, HIDDEN_SIZES, 10)
classical_device = "cuda" 
adam_optimizer = torch.optim.Adam(adam_model.parameters(), 
                             lr=0.01,
                             betas=[0.9, 0.999],
                             )

In [8]:
train(
    model=adam_model,
    train_loader=train_loader,
    test_loader=test_loader,
    loss_fn=loss_fn,
    optimizer=adam_optimizer,
    c_device=classical_device,
    epochs=NUM_EPOCHS, 
    experiment_name = EXPERIMENT_NAME,
    run_name="adam-optimizer-cuda"
)

Epoch 000 | train_loss=1.9697 | train_acc=0.414 | test_loss=1.9650 | test_acc=0.423 | 
Epoch 005 | train_loss=0.9307 | train_acc=0.814 | test_loss=0.9214 | test_acc=0.825 | 
Epoch 010 | train_loss=0.5254 | train_acc=0.876 | test_loss=0.5170 | test_acc=0.880 | 
Epoch 015 | train_loss=0.3737 | train_acc=0.898 | test_loss=0.3651 | test_acc=0.903 | 
Epoch 020 | train_loss=0.3032 | train_acc=0.912 | test_loss=0.2982 | test_acc=0.916 | 
Epoch 025 | train_loss=0.2586 | train_acc=0.925 | test_loss=0.2582 | test_acc=0.925 | 


2026/05/02 18:04:46 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


Epoch 029 | train_loss=0.2319 | train_acc=0.933 | test_loss=0.2361 | test_acc=0.932 | 


{'run_id': '5b1cc33de1db487081464254a2e376ae',
 'experiment_id': '467419381031699499',
 'experiment_name': 'optimizer-comparison-mlp-MNIST-q',
 'tracking_uri': 'file:///home/filip/studia/master/second-order-by-annealer/mlruns',
 'warm_start_epochs': 1,
 'optimization_time_sec': 0.9997913790066377,
 'raw_optimization_time_sec': 1.1128408330059756,
 'sum_epoch_optimization_time_sec': 0.9997913790066377,
 'train_evaluation_time_sec': 119.37167075399702,
 'test_evaluation_time_sec': 18.332781634999264,
 'evaluation_time_sec': 137.7044523889963,
 'training_time_sec': 258.1519421689991,
 'final_train_loss': 0.23194216191768646,
 'final_test_loss': 0.23614992201328278,
 'final_train_metric': 0.9335,
 'final_test_metric': 0.9317,
 'optimizer': 'Adam',
 'seed': None,
 'epochs_completed': 30,
 'early_stopped': False,
 'early_stop_reason': None,
 'zero_acceptance_patience': 6,
 'final_zero_acceptance_streak': 0}

## Optimization using LBFGS-style second-order optimizer

In [9]:
loss_fn = nn.CrossEntropyLoss()
lbfgs_model = QuadraticMLP(784, HIDDEN_SIZES, 10)
classical_device = "cpu" 
lbfgs_optimizer = torch.optim.LBFGS(lbfgs_model.parameters(), 
                              lr=0.01,
                              max_iter=1,
                              history_size=100,
                              line_search_fn=None,
                              )

In [10]:
train(
    model=lbfgs_model,
    train_loader=train_loader,
    test_loader=test_loader,
    loss_fn=loss_fn,
    optimizer=lbfgs_optimizer,
    c_device=classical_device,
    epochs=NUM_EPOCHS,
    experiment_name = EXPERIMENT_NAME,
    run_name="lbfgs-optimizer"
)

Epoch 000 | train_loss=2.3180 | train_acc=0.050 | test_loss=2.3170 | test_acc=0.049 | 
Epoch 005 | train_loss=2.0236 | train_acc=0.637 | test_loss=2.0173 | test_acc=0.645 | 
Epoch 010 | train_loss=1.7860 | train_acc=0.705 | test_loss=1.7763 | test_acc=0.718 | 
Epoch 015 | train_loss=1.5397 | train_acc=0.734 | test_loss=1.5272 | test_acc=0.747 | 
Epoch 020 | train_loss=1.3041 | train_acc=0.760 | test_loss=1.2896 | test_acc=0.774 | 
Epoch 025 | train_loss=1.1271 | train_acc=0.782 | test_loss=1.1111 | test_acc=0.795 | 


2026/05/02 18:09:03 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


Epoch 029 | train_loss=1.0283 | train_acc=0.798 | test_loss=1.0115 | test_acc=0.810 | 


{'run_id': 'eb7fe8e47595404aae70e9af668e480a',
 'experiment_id': '467419381031699499',
 'experiment_name': 'optimizer-comparison-mlp-MNIST-q',
 'tracking_uri': 'file:///home/filip/studia/master/second-order-by-annealer/mlruns',
 'warm_start_epochs': 1,
 'optimization_time_sec': 2.455077745999006,
 'raw_optimization_time_sec': 2.5559086439989187,
 'sum_epoch_optimization_time_sec': 2.455077745999006,
 'train_evaluation_time_sec': 117.19366824200006,
 'test_evaluation_time_sec': 18.586317775005227,
 'evaluation_time_sec': 135.77998601700529,
 'training_time_sec': 254.18063176500073,
 'final_train_loss': 1.0282586812973022,
 'final_test_loss': 1.011487603187561,
 'final_train_metric': 0.79845,
 'final_test_metric': 0.81,
 'optimizer': 'LBFGS',
 'seed': None,
 'epochs_completed': 30,
 'early_stopped': False,
 'early_stop_reason': None,
 'zero_acceptance_patience': 6,
 'final_zero_acceptance_streak': 0}

## Optimization using Newton-Rapson method

In [11]:
loss_fn = nn.CrossEntropyLoss() 
newton_model = QuadraticMLP(784, HIDDEN_SIZES, 10)
classical_device = "cpu"
newton_optimizer = NewtonOptimizer(newton_model.parameters(), 
                                   lr=1, 
                                   max_iter=1,
                                   damping=1e-4,
                                   )

In [12]:
train(
    model=newton_model,
    train_loader=train_loader,
    test_loader=test_loader,
    loss_fn=loss_fn,
    optimizer=newton_optimizer,
    c_device=classical_device,
    epochs=NUM_EPOCHS,
    experiment_name = EXPERIMENT_NAME,
    run_name="newton-optimizer"
)

RuntimeError: [enforce fail at alloc_cpu.cpp:127] err == 0. DefaultCPUAllocator: can't allocate memory: you tried to allocate 44047935376 bytes. Error code 12 (Cannot allocate memory)

## Optimization using Quadratic Annealer (cpu + momentum)

In [20]:
loss_fn = nn.CrossEntropyLoss()
model = QuadraticMLP(784, HIDDEN_SIZES, 10)
classical_device = "cpu" 
        
sampler = build_sampler(mode="simulated")

optimizer = QuadraticAnnealingOptimizer(
    sampler=sampler,
    model=model,
    device="cpu",
    subset_size=48,
    step_size=0.05,
    num_reads=100,
    beta=0.9,
)

In [21]:
train(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    loss_fn=loss_fn,
    optimizer=optimizer,
    c_device=classical_device,
    epochs=NUM_EPOCHS,
    experiment_name = EXPERIMENT_NAME,
    run_name="quadratic-annealer-optimizer-256"
)

Epoch 000 | train_loss=2.2978 | train_acc=0.109 | test_loss=2.2981 | test_acc=0.107 | 
Epoch 005 | train_loss=1.9659 | train_acc=0.280 | test_loss=1.9549 | test_acc=0.283 | 
Epoch 010 | train_loss=1.6532 | train_acc=0.426 | test_loss=1.6432 | test_acc=0.429 | 
Epoch 015 | train_loss=1.5392 | train_acc=0.487 | test_loss=1.5257 | test_acc=0.493 | 
Epoch 020 | train_loss=1.5392 | train_acc=0.487 | test_loss=1.5257 | test_acc=0.493 | 


2026/05/02 19:07:01 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


{'run_id': '3edd3cf5e5e24c9ea88d6f9bd3accdbc',
 'experiment_id': '467419381031699499',
 'experiment_name': 'optimizer-comparison-mlp-MNIST-q',
 'tracking_uri': 'file:///home/filip/studia/master/second-order-by-annealer/mlruns',
 'warm_start_epochs': 1,
 'optimization_time_sec': 126.3422011299881,
 'raw_optimization_time_sec': 132.06730426398644,
 'sum_epoch_optimization_time_sec': 126.3422011299881,
 'train_evaluation_time_sec': 87.17361740099295,
 'test_evaluation_time_sec': 14.785538580988941,
 'evaluation_time_sec': 101.9591559819819,
 'training_time_sec': 322.7842162459965,
 'final_train_loss': 1.539155125617981,
 'final_test_loss': 1.5257139205932617,
 'final_train_metric': 0.48688333333333333,
 'final_test_metric': 0.4926,
 'optimizer': 'QuadraticAnnealingOptimizer',
 'seed': None,
 'epochs_completed': 22,
 'early_stopped': True,
 'early_stop_reason': 'acceptance_rate_was_zero_for_6_consecutive_epochs',
 'zero_acceptance_patience': 6,
 'final_zero_acceptance_streak': 6}

## Optimization using Quadratic Annealer (D-Wave + momentum)

In [18]:
loss_fn = nn.CrossEntropyLoss()
model = QuadraticMLP(784, HIDDEN_SIZES, 10)
classical_device = "cpu" 

sampler = build_sampler(mode="dwave")

optimizer = QuadraticAnnealingOptimizer(
    sampler=sampler,
    model=model,
    device="cpu",
    subset_size=48,
    step_size=0.05,
    num_reads=100,
    beta=0.9,
)

In [19]:
train(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    loss_fn=loss_fn,
    optimizer=optimizer,
    c_device=classical_device,
    epochs=NUM_EPOCHS,
    experiment_name = EXPERIMENT_NAME,
    run_name="quadratic-annealer-dwave-optimizer-48"
)

Epoch 000 | train_loss=2.2859 | train_acc=0.206 | test_loss=2.2868 | test_acc=0.213 | 
Epoch 005 | train_loss=1.9985 | train_acc=0.251 | test_loss=1.9968 | test_acc=0.246 | 
Epoch 010 | train_loss=1.8460 | train_acc=0.342 | test_loss=1.8424 | test_acc=0.347 | 
Epoch 015 | train_loss=1.7856 | train_acc=0.395 | test_loss=1.7807 | test_acc=0.396 | 


2026/05/02 18:23:54 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


{'run_id': 'dd187e5a1eec4c7db2fb0c521cc6d090',
 'experiment_id': '467419381031699499',
 'experiment_name': 'optimizer-comparison-mlp-MNIST-q',
 'tracking_uri': 'file:///home/filip/studia/master/second-order-by-annealer/mlruns',
 'warm_start_epochs': 1,
 'optimization_time_sec': 100.12996197699977,
 'raw_optimization_time_sec': 109.55645658499998,
 'sum_epoch_optimization_time_sec': 100.12996197699977,
 'train_evaluation_time_sec': 73.59660395500032,
 'test_evaluation_time_sec': 12.556252305996168,
 'evaluation_time_sec': 86.15285626099649,
 'training_time_sec': 306.992198033,
 'final_train_loss': 1.7856335639953613,
 'final_test_loss': 1.7807289361953735,
 'final_train_metric': 0.39531666666666665,
 'final_test_metric': 0.3961,
 'optimizer': 'QuadraticAnnealingOptimizer',
 'seed': None,
 'epochs_completed': 19,
 'early_stopped': True,
 'early_stop_reason': 'acceptance_rate_was_zero_for_6_consecutive_epochs',
 'zero_acceptance_patience': 6,
 'final_zero_acceptance_streak': 6}